# Day 1 - Data Science Lifecycle: From Question to Decision

One notebook. One dataset. One complete journey through the Data Science lifecycle.

```
1. Problem Understanding
2. Data Collection
3. Data Understanding
4. Data Cleaning
5. Exploratory Data Analysis
6. Statistics
7. Visualization
8. Feature Preparation
9. Communication and Decision
```

**Rule:** Start with the question, not the tool.

---
## Step 0: Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, MinMaxScaler

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50)

print("Libraries ready.")

---
## Step 1: Problem Understanding

Before touching data, answer these:

- **What problem?** Understand customer activity patterns
- **Who needs the answer?** Business team
- **What decision?** Which customer groups need attention
- **What action?** Targeted engagement strategies

**Business question:** How can we improve customer engagement?

**Data questions:**

In [ ]:
questions = [
    "Which membership group spends the most?",
    "Which city has the most customers?",
    "What is a typical monthly spend?",
    "Are visits and spending related?",
    "Which customers may need attention?",
]

for q in questions:
    print("-", q)

---
## Step 2: Data Collection

Data can come from CSV, databases, APIs, surveys, etc.

For this demo, we create a small customer activity dataset with **intentional problems** (messy text, missing values, duplicates, invalid numbers).

In [ ]:
raw = pd.DataFrame({
    "customer_id": [101, 102, 103, 104, 105, 106, 106, 107, 108, 109],
    "name":        ["Asha", "Ravi", "Meera", "John", "Fatima", "Chen", "Chen", "Sara", "Vikram", "Nina"],
    "city":        [" Mumbai", "delhi", "Chennai", "Mumbai", np.nan, "chennai", "chennai", "Delhi ", "Mumbai", "Delhi"],
    "membership":  ["Gold", "Silver", "Gold", "Bronze", "Silver", "gold", "gold", np.nan, "Bronze", "Silver"],
    "monthly_spend":    [1200, 850, 1450, 700, np.nan, 1600, 1600, 950, -200, 1100],
    "visits_per_month": [4, 3, 5, 2, 6, 5, 5, np.nan, 1, 4],
    "signup_month":     ["Jan", "Feb", "Feb", "Mar", "Apr", "Apr", "Apr", "May", "Jun", "Jun"],
})

raw

---
## Step 3: Data Understanding

Before cleaning, **look first**. Understand rows, columns, types, and values.

In [ ]:
# Shape and columns
print("Shape:", raw.shape)
print("Columns:", list(raw.columns))

In [ ]:
# Data types and non-null counts
raw.info()

In [ ]:
# Quick look at first and last rows
raw.head()

In [ ]:
# Unique values in text columns - spot inconsistencies
for col in ["city", "membership"]:
    print(f"{col}: {raw[col].unique()}")

**Problems spotted:**
- Messy text: `delhi`, `Delhi `, ` Mumbai`, `gold`
- Missing values: city, membership, monthly_spend, visits_per_month
- Duplicate row: customer 106 appears twice
- Invalid value: negative monthly_spend (-200)

---
## Step 4: Data Cleaning

**Rule:** Always work on a copy. Never overwrite raw data.

In [ ]:
df = raw.copy()
print("Working copy created. Shape:", df.shape)

### 4a. Fix messy text

In [ ]:
df["city"] = df["city"].str.strip().str.title()
df["membership"] = df["membership"].str.strip().str.title()

print("city:", df["city"].unique())
print("membership:", df["membership"].unique())

### 4b. Remove duplicates

In [ ]:
before = len(df)
df = df.drop_duplicates()
print(f"Removed {before - len(df)} duplicate row(s). Rows now: {len(df)}")

### 4c. Handle missing values

In [ ]:
print("Missing values before:")
print(df.isna().sum())

In [ ]:
# Numeric: fill with median
spend_median = df["monthly_spend"].median()
visits_median = df["visits_per_month"].median()

df["monthly_spend"] = df["monthly_spend"].fillna(spend_median)
df["visits_per_month"] = df["visits_per_month"].fillna(visits_median)

# Categorical: fill with "Unknown"
df["city"] = df["city"].fillna("Unknown")
df["membership"] = df["membership"].fillna("Unknown")

print(f"Spend median used: {spend_median}")
print(f"Visits median used: {visits_median}")
print(f"\nMissing values after:\n{df.isna().sum()}")

### 4d. Fix invalid values

In [ ]:
# Find and fix negative spend
invalid = df[df["monthly_spend"] < 0]
print("Invalid rows:")
display(invalid)

df.loc[df["monthly_spend"] < 0, "monthly_spend"] = spend_median
print(f"\nReplaced with median: {spend_median}")

### 4e. Final quality check

In [ ]:
print("Rows:", len(df))
print("Missing:", df.isna().sum().sum())
print("Duplicates:", df.duplicated().sum())
print("Negative spend:", (df["monthly_spend"] < 0).sum())
print()
df

---
## Step 5: Exploratory Data Analysis (EDA)

EDA = conversation between analyst and data.

We explore patterns using selection, filtering, sorting, grouping.

In [ ]:
# How many customers per city?
df["city"].value_counts()

In [ ]:
# How many customers per membership?
df["membership"].value_counts()

In [ ]:
# Top spenders
df.sort_values("monthly_spend", ascending=False).head(3)

In [ ]:
# Filter: high-spend customers (above 1000)
df[df["monthly_spend"] > 1000]

In [ ]:
# Average spend by membership
df.groupby("membership")["monthly_spend"].mean().sort_values(ascending=False).round(2)

In [ ]:
# Average spend by city
df.groupby("city")["monthly_spend"].mean().sort_values(ascending=False).round(2)

---
## Step 6: Statistics

Statistics summarizes data without reading every row.

- **Center:** mean, median, mode
- **Spread:** range, variance, standard deviation, IQR
- **Relationship:** correlation

### 6a. Center: Mean, Median, Mode

In [ ]:
print("Mean spend:", df["monthly_spend"].mean().round(2))
print("Median spend:", df["monthly_spend"].median())
print("Mode city:", df["city"].mode()[0])
print("Mode membership:", df["membership"].mode()[0])

### Mean vs Median

Mean moves with extreme values. Median stays stable.

In [ ]:
normal = [700, 850, 1200, 1450, 2200]
extreme = [700, 850, 1200, 1450, 8000]

pd.DataFrame({
    "case": ["normal", "with extreme value"],
    "mean": [np.mean(normal), np.mean(extreme)],
    "median": [np.median(normal), np.median(extreme)],
})

### 6b. Spread: Variance, Std Dev, IQR

Average alone is not enough. Two groups can have the same mean but different spread.

In [ ]:
# Same mean, different spread
group_a = np.array([900, 1000, 1100])
group_b = np.array([200, 1000, 1800])

print(f"Group A: mean={group_a.mean()}, std={group_a.std():.1f}")
print(f"Group B: mean={group_b.mean()}, std={group_b.std():.1f}")

**Why squared differences?** Raw differences cancel out (negative + positive = 0). Squaring keeps them positive.

In [ ]:
values = np.array([8, 10, 12])
diffs = values - values.mean()

pd.DataFrame({
    "value": values,
    "diff_from_mean": diffs,
    "squared_diff": diffs ** 2,
})

In [ ]:
print("Sum of raw diffs:", diffs.sum(), "(cancels out!)")
print("Sum of squared diffs:", (diffs ** 2).sum(), "(useful!)")
print()
print("Variance = mean of squared diffs =", (diffs ** 2).mean())
print("Std dev = sqrt of variance =", np.sqrt((diffs ** 2).mean()).round(3))

In [ ]:
# describe() - fast summary dashboard
df[["monthly_spend", "visits_per_month"]].describe().round(2)

### 6c. IQR and Outlier Check

In [ ]:
q1 = df["monthly_spend"].quantile(0.25)
q3 = df["monthly_spend"].quantile(0.75)
iqr = q3 - q1
lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr

print(f"Q1={q1}, Q3={q3}, IQR={iqr}")
print(f"Fences: [{lower}, {upper}]")
print(f"Outliers: {len(df[(df['monthly_spend'] < lower) | (df['monthly_spend'] > upper)])} rows")

### 6d. Correlation

Do visits and spending move together?

- Close to **+1**: strong same direction
- Close to **-1**: strong opposite direction
- Close to **0**: no clear linear pattern

**Correlation does not prove cause.**

In [ ]:
corr = df[["monthly_spend", "visits_per_month"]].corr().round(3)
corr

In [ ]:
r = df["visits_per_month"].corr(df["monthly_spend"])
strength = "weak" if abs(r) < 0.3 else "moderate" if abs(r) < 0.7 else "strong"
direction = "positive" if r > 0 else "negative"
print(f"Correlation: {r:.3f} -> {strength} {direction} linear relationship")

### 6e. Grouped statistics

In [ ]:
df.groupby("membership")["monthly_spend"].agg(
    count="count", mean="mean", median="median", std="std"
).round(2).sort_values("mean", ascending=False)

---
## Step 7: Visualization

Charts answer questions visually.

| Question | Chart |
|---|---|
| Compare groups | Bar chart |
| See distribution | Histogram |
| Compare spread | Box plot |
| See relationship | Scatter plot |
| Count categories | Count plot |

### 7a. Bar chart: Average spend by membership

In [ ]:
membership_spend = df.groupby("membership")["monthly_spend"].mean().sort_values(ascending=False)

plt.figure(figsize=(7, 4))
membership_spend.plot(kind="bar", color="#2F80ED")
plt.title("Average Monthly Spend by Membership")
plt.xlabel("Membership")
plt.ylabel("Average Monthly Spend")
plt.xticks(rotation=0)
plt.show()

print(f"Highest: {membership_spend.idxmax()} at {membership_spend.max():.0f}")

### 7b. Count plot: Customers per city

In [ ]:
plt.figure(figsize=(7, 4))
sns.countplot(data=df, x="city", order=df["city"].value_counts().index, color="#F2994A")
plt.title("Customer Count by City")
plt.xlabel("City")
plt.ylabel("Count")
plt.show()

### 7c. Histogram: Monthly spend distribution

In [ ]:
plt.figure(figsize=(7, 4))
plt.hist(df["monthly_spend"], bins=5, color="#9B51E0", edgecolor="white")
plt.title("Distribution of Monthly Spend")
plt.xlabel("Monthly Spend")
plt.ylabel("Number of Customers")
plt.show()

### 7d. Box plot: Spend spread by membership

In [ ]:
plt.figure(figsize=(7, 4))
sns.boxplot(data=df, x="membership", y="monthly_spend", color="#56CCF2")
sns.stripplot(data=df, x="membership", y="monthly_spend", color="#333", size=7, jitter=True)
plt.title("Monthly Spend by Membership")
plt.xlabel("Membership")
plt.ylabel("Monthly Spend")
plt.show()

### 7e. Scatter plot: Visits vs Spend

In [ ]:
plt.figure(figsize=(7, 4))
sns.scatterplot(data=df, x="visits_per_month", y="monthly_spend", hue="membership", s=90)
plt.title("Visits per Month vs Monthly Spend")
plt.xlabel("Visits per Month")
plt.ylabel("Monthly Spend")
plt.show()

print(f"Correlation: {r:.3f} - association, not cause.")

### 7f. Line chart: Spend trend by signup month

In [ ]:
month_order = ["Jan", "Feb", "Mar", "Apr", "May", "Jun"]
trend = df.groupby("signup_month")["monthly_spend"].mean().reindex(month_order)

plt.figure(figsize=(7, 4))
plt.plot(trend.index, trend.values, marker="o", color="#27AE60")
plt.title("Average Spend by Signup Month")
plt.xlabel("Signup Month")
plt.ylabel("Average Monthly Spend")
plt.show()

---
## Step 8: Feature Preparation

Prepare columns for later analysis or modeling.

- **Scaling:** put numeric columns on comparable scales
- **Encoding:** convert text categories to numbers

### 8a. New feature: spend per visit

In [ ]:
df["spend_per_visit"] = (df["monthly_spend"] / df["visits_per_month"]).round(2)
df[["name", "monthly_spend", "visits_per_month", "spend_per_visit"]]

### 8b. Standard Scaling

Result: mean near 0, std near 1. Tells how far each value is from average.

In [ ]:
num_cols = ["monthly_spend", "visits_per_month"]

scaler = StandardScaler()
scaled = pd.DataFrame(
    scaler.fit_transform(df[num_cols]),
    columns=[c + "_scaled" for c in num_cols]
).round(3)

scaled

In [ ]:
# Verify: mean ~ 0, std ~ 1
scaled.describe().round(3)

### 8c. Min-Max Normalization

Result: values between 0 and 1. 0 = minimum, 1 = maximum.

In [ ]:
normalizer = MinMaxScaler()
normalized = pd.DataFrame(
    normalizer.fit_transform(df[num_cols]),
    columns=[c + "_norm" for c in num_cols]
).round(3)

normalized

### Scaling vs Normalization

| Method | Result | Question it answers |
|---|---|---|
| Standard scaling | Mean~0, Std~1 | How far from average? |
| Min-max normalization | 0 to 1 | Where between min and max? |

### 8d. One-Hot Encoding

Convert text categories into 0/1 columns. Use when categories have **no natural order**.

In [ ]:
city_encoded = pd.get_dummies(df["city"], prefix="city", dtype=int)
city_encoded

### 8e. Ordered Mapping

Use when categories **have a real order** (Bronze < Silver < Gold).

In [ ]:
level_map = {"Bronze": 1, "Silver": 2, "Gold": 3, "Unknown": 0}
df["membership_level"] = df["membership"].map(level_map)

df[["name", "membership", "membership_level"]]

### Encoding decision guide

| Situation | Method |
|---|---|
| No natural order (city) | One-hot encoding |
| Meaningful order (Bronze < Silver < Gold) | Ordered mapping |
| IDs and names | Keep for reference, don't encode |

---
## Step 9: Communication and Decision

Data Science is incomplete without communicating findings.

A good report has: **findings + evidence + caution + recommendation**

In [ ]:
# Summary table
summary = pd.DataFrame({
    "item": [
        "total_customers",
        "top_city",
        "top_membership_by_spend",
        "mean_monthly_spend",
        "median_monthly_spend",
        "visits_spend_correlation",
    ],
    "value": [
        len(df),
        df["city"].value_counts().idxmax(),
        membership_spend.idxmax(),
        round(df["monthly_spend"].mean(), 2),
        df["monthly_spend"].median(),
        round(r, 3),
    ],
})

summary

In [ ]:
# Final report
report = f"""DAY 1 MINI EDA REPORT
{'='*40}

DATASET: {len(df)} customers, {df.shape[1]} columns (cleaned)

FINDINGS:
1. {df['city'].value_counts().idxmax()} has the most customers ({df['city'].value_counts().max()}).
2. {membership_spend.idxmax()} membership has highest avg spend ({membership_spend.max():.0f}).
3. Median monthly spend is {df['monthly_spend'].median():.0f} (half spend at or below this).
4. Visits and spend correlation: {r:.3f} ({strength} {direction}).

RECOMMENDATION:
- Investigate {membership_spend.idxmax()} customers to understand high-spend behavior.
- Look into low-visit customers for engagement opportunities.

CAUTION:
- Small dataset ({len(df)} rows). Validate with more data before business decisions.
- Correlation is not causation.
"""

print(report)

---
## Lifecycle Recap

| Step | What we did | Key tool |
|---|---|---|
| 1. Problem Understanding | Framed business and data questions | Thinking |
| 2. Data Collection | Created dataset with real-world problems | `pd.DataFrame()` |
| 3. Data Understanding | Inspected shape, types, values | `.shape`, `.info()`, `.unique()` |
| 4. Data Cleaning | Fixed text, duplicates, missing, invalid | `.str.strip()`, `.fillna()`, `.drop_duplicates()` |
| 5. EDA | Filtered, sorted, grouped | `.value_counts()`, `.groupby()` |
| 6. Statistics | Center, spread, correlation | `.mean()`, `.std()`, `.corr()` |
| 7. Visualization | Bar, histogram, box, scatter, line | `matplotlib`, `seaborn` |
| 8. Feature Preparation | Scaling, encoding, new features | `StandardScaler`, `get_dummies()`, `.map()` |
| 9. Communication | Report with findings + evidence + caution | Plain English |

**Next:** Day 2 - Predicting numbers from data (regression)